In [ ]:
import requests
import json
from spellchecker import SpellChecker

# URL to the JSON file
json_url = "https://raw.githubusercontent.com/dwyl/english-words/master/words_dictionary.json"

# Download and load the JSON file
response = requests.get(json_url)
words_dict = response.json()  # This loads the JSON as a dictionary

# Initialize the spell checker with its default dictionary
spell = SpellChecker()

# Add the words from the JSON dictionary to the spell checker's known words
for word in words_dict.keys():
    spell.word_frequency.add(word)


In [ ]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io

spell = SpellChecker()

def correct_ocr_errors(text):
    corrected_text = []
    words = text.split()
    for word in words:
        corrected_word = spell.correction(word)
        if corrected_word is None:  # Handle NoneType returned by spell.correction
            corrected_word = word
        corrected_text.append(corrected_word)
    return ' '.join(corrected_text)

def pdf_to_ocr(pdf_path):
    pdf_document = fitz.open(pdf_path)
    ocr_text = ""
    custom_config = r'--oem 3 --psm 6'
    
    for page_num in range(len(pdf_document)):
        page = pdf_document.load_page(page_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))  # Increase resolution
        
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert('L')
        img = img.point(lambda x: 0 if x < 128 else 255, '1')  # Binarize image
        
        text = pytesseract.image_to_string(img, config=custom_config)
        
        text = correct_ocr_errors(text)
        
        ocr_text += f"Page {page_num + 1}:\n{text}\n"
        
        print(f"Processed page {page_num + 1}/{len(pdf_document)}")
    
    return ocr_text

# Usage
pdf_path = '/content/2.pdf'  # Replace with your file's actual path
ocr_text = pdf_to_ocr(pdf_path)

# Save the OCR result to a text file
with open("output_ocr.txt", "w") as text_file:
    text_file.write(ocr_text)

# Optional: Download the output text file
from google.colab import files
files.download("output_ocr.txt")
